In [4]:
import os
import sys

current_dir = os.path.dirname(os.path.abspath("Incremental-Learning-MycroftMinds"))
parent_dir = os.path.dirname(current_dir)
# Insert the parent directory at the beginning of sys.path if it's not already there.
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [5]:
import time
from typing import Optional, Tuple, Dict, Any, List

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from river.forest import ARFRegressor
#from river.ensemble import BaggingRegressor, SRPRegressor
from river.tree import HoeffdingTreeRegressor, HoeffdingAdaptiveTreeRegressor
#from river.drift.binary import DDM, EDDM
from src.bagging import ARFRegressorFODT
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

from src.data.io import load_dataset
from src.utils.memory import calculate_memory_usage
from src.utils.utils import get_device_data
from src.trees import fastodt
from src.boosting import IncubationBoost
from src.ensemble_methods import elasticGBDT
from src.evaluation.metrics import calculate_metrics, all_time_aggregation_metrics
from src.evaluation.visualization import plot_part_predictions, plot_hourly_predictions
from src.evaluation.training import all_valid_splits_seq_fit_predict, test_set_seq_fit_predict
from src.evaluation.baseline import PreviousValueRegressor
from src.evaluation.tuning import grid_search_tuning
from src.data import dataloader

import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)

In [7]:
lag_args = {
    #'lag_5m':'5M',
    'lag_1h':'1H',
    'lag_24h':'24H',
    }

data_dir = '../data/first_batch'

columns_to_add_lag = [
            'L1_act_power', 'L1_app_power', 'L1_current', 'L1_react_power', 'L1_voltage',
            'L2_act_power', 'L2_app_power', 'L2_current', 'L2_react_power', 'L2_voltage',
            'L3_act_power', 'L3_app_power', 'L3_current', 'L3_react_power', 'L3_voltage',
            'grid_freq', 'powerfactor1', 'powerfactor2', 'powerfactor3'
        ]
device_list = ['df_TRB145-3',
               #'df_TRB145-5',
               ]

# target
target_feature = 'L2_act_power'

train, valid, test = dataloader.get_train_test_sets(
    data_dir=data_dir,
    lag_freqs=lag_args,
    columns_to_add_lag=columns_to_add_lag,
    target_feature=target_feature,
    device_list=device_list,
    walk_forward_sets=True,
    train_days=28,
    valid_days=28,
    optimize_memory=True,  # Enable memory optimization
    save=True,  # Save processed data for faster future loading
    intermediate_prefix='dev_3_hourly_wfv',  # Prefix for saved files
    time_agg='1H',  # Time aggregation frequency
    new_data=True,
    fill_missing=True
)

[INFO] Aggregating data by: 1H
input_features:  ['time', 'device_id', 'hour', 'weekday', 'month', 'L1_act_power_lag_1h', 'L1_act_power_lag_24h', 'L1_app_power_lag_1h', 'L1_app_power_lag_24h', 'L1_current_lag_1h', 'L1_current_lag_24h', 'L1_react_power_lag_1h', 'L1_react_power_lag_24h', 'L1_voltage_lag_1h', 'L1_voltage_lag_24h', 'L2_act_power_lag_1h', 'L2_act_power_lag_24h', 'L2_app_power_lag_1h', 'L2_app_power_lag_24h', 'L2_current_lag_1h', 'L2_current_lag_24h', 'L2_react_power_lag_1h', 'L2_react_power_lag_24h', 'L2_voltage_lag_1h', 'L2_voltage_lag_24h', 'L3_act_power_lag_1h', 'L3_act_power_lag_24h', 'L3_app_power_lag_1h', 'L3_app_power_lag_24h', 'L3_current_lag_1h', 'L3_current_lag_24h', 'L3_react_power_lag_1h', 'L3_react_power_lag_24h', 'L3_voltage_lag_1h', 'L3_voltage_lag_24h', 'grid_freq_lag_1h', 'grid_freq_lag_24h', 'powerfactor1_lag_1h', 'powerfactor1_lag_24h', 'powerfactor2_lag_1h', 'powerfactor2_lag_24h', 'powerfactor3_lag_1h', 'powerfactor3_lag_24h']

Overall data time range:  

In [8]:
HOURLY = True
DATA_FREQUENCY = 'H' if HOURLY else '5min'

# Measure loading time
start_time = time.time()

# Load data
if HOURLY:
    train, valid, test = load_dataset(data_dir=data_dir, prefix='dev_3_hourly_wfv', use_numpy=True)
    features = ['time', 'device_id', 'hour', 'weekday', 'month',
                 'L1_act_power_lag_1h', 'L1_act_power_lag_24h', 'L1_app_power_lag_1h', 'L1_app_power_lag_24h',
                 'L1_current_lag_1h', 'L1_current_lag_24h', 'L1_react_power_lag_1h', 'L1_react_power_lag_24h',
                 'L1_voltage_lag_1h', 'L1_voltage_lag_24h',
                 'L2_act_power_lag_1h', 'L2_act_power_lag_24h', 'L2_app_power_lag_1h', 'L2_app_power_lag_24h',
                 'L2_current_lag_1h', 'L2_current_lag_24h', 'L2_react_power_lag_1h', 'L2_react_power_lag_24h',
                 'L2_voltage_lag_1h', 'L2_voltage_lag_24h',
                 'L3_act_power_lag_1h', 'L3_act_power_lag_24h', 'L3_app_power_lag_1h', 'L3_app_power_lag_24h',
                 'L3_current_lag_1h', 'L3_current_lag_24h', 'L3_react_power_lag_1h', 'L3_react_power_lag_24h',
                 'L3_voltage_lag_1h', 'L3_voltage_lag_24h',
                 'grid_freq_lag_1h', 'grid_freq_lag_24h',
                 'powerfactor1_lag_1h', 'powerfactor1_lag_24h',
                 'powerfactor2_lag_1h', 'powerfactor2_lag_24h',
                 'powerfactor3_lag_1h', 'powerfactor3_lag_24h']
else:
    train, valid, test = load_dataset(data_dir=data_dir, prefix='8_dev_5_min_wfv', use_numpy=True)
    # copy features from dataloader printouts
    features = []

loading_time = time.time() - start_time
print(f"Data loaded in {loading_time/60:.2f} minutes")

[INFO] Loading train data...
[INFO] Train data loaded: 3 samples
[INFO] Loading valid data...
[INFO] Valid data loaded: 3 samples
[INFO] Loading test data...
[INFO] Test data loaded: 2 samples
Data loaded in 0.00 minutes


### HoeffdingTree

In [90]:
update_freq = "1h"

model = HoeffdingTreeRegressor

param_grid = {
    'max_depth': np.arange(2, 11, 3),
    'grace_period': np.arange(5, 35, 5),
    'tau': np.arange(0.01, 0.1, 0.03),
    'delta': [0.5],
}

best_params, best_scores = grid_search_tuning(param_grid, model, train, valid)

training with: {'max_depth': np.int64(2), 'grace_period': np.int64(5), 'tau': np.float64(0.01), 'delta': 0.5}


Processing splits:   0%|          | 0/3 [00:00<?, ?it/s]

Processing devices in split 0:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-09-14 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 1:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-10-12 08:00:00
Train predictions completed.
Validation predictions completed.


... (495 outputs left out)

Processing devices in split 1:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-10-12 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 2:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-11-09 08:00:00
Train predictions completed.
Validation predictions completed.


In [98]:
print(best_params)
print(best_scores)

{'rmse': {'max_depth': np.int64(5), 'grace_period': np.int64(30), 'tau': np.float64(0.01), 'delta': 0.5}, 'mae': {'max_depth': np.int64(5), 'grace_period': np.int64(30), 'tau': np.float64(0.06999999999999999), 'delta': 0.5}, 'mape': {'max_depth': np.int64(5), 'grace_period': np.int64(25), 'tau': np.float64(0.06999999999999999), 'delta': 0.5}}
{'rmse': 0.3444369684330996, 'mae': 0.17773947067320114, 'mape': 0.42482268810146556}


In [93]:
best_scores

{'rmse': 0.3444369684330996,
 'mae': 0.17773947067320114,
 'mape': 0.42482268810146556}

In [100]:
best_model = HoeffdingTreeRegressor(**best_params['rmse'])

hoef_all_train_preds, hoef_all_valid_preds, \
hoef_all_train_metrics, hoef_all_valid_metrics = all_valid_splits_seq_fit_predict(
    model=best_model,
    train=train,
    valid=valid,
    incremental=True,
    data_freq=DATA_FREQUENCY,
    update_freq=update_freq,
    start_split=0,
    #end_split=4,  # Use only the third and fourth splits
    verbose=False
)

print("\n\nTrain period metrics:")
hoef_all_time_train_metrics = all_time_aggregation_metrics(hoef_all_train_metrics, data_freq=DATA_FREQUENCY, verbose=False)
print("\n\nValid period metrics:")
hoef_all_time_valid_metrics = all_time_aggregation_metrics(hoef_all_valid_metrics, data_freq=DATA_FREQUENCY, verbose=True)

Processing splits:   0%|          | 0/3 [00:00<?, ?it/s]

Processing devices in split 0:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-09-14 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 1:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-10-12 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 2:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-11-09 08:00:00
Train predictions completed.
Validation predictions completed.


Train period metrics:


Valid period metrics:


Aggregated Metrics Across Periods
Device ID: df_TRB145-3
--------------------------------------------------
Hourly Metrics:
	RMSE: 0.3444 ± 0.0492
	MAE: 0.1785 ± 0.0173
	MAPE: 45.8416% ± 8.2874%

	Directional Metrics:
		Directional Accuracy: 48.7535% ± 0.5118%
		First-Diff Correlation: 0.4778 ± 0.0386

	Immediate Detection (delta=0):
		Spike Precision: 0.0495 ± 0.0350
		Spike Recall: 0.0312 ± 0.0274
		Spike F1: 0.0363 ± 0.0283
		Drop Precision: 0.0534 ± 0.0379
		Drop Recall: 0.0331 ± 0.0294
		Drop F1: 0.0386 ± 0.0303

	1-Hour Delayed Detection (delta=1):
		Spike Precision: 0.3172 ± 0.1051
		Spike Recall: 0.2160 ± 0.0517
		Spike F1: 0.2425 ± 0.0302
		Drop Precision: 0.2079 ± 0.0466
		Drop Recall: 0.1486 ± 0.0535
		Drop F1: 0.1642 ± 0.0419
Daily Metrics:
	RMSE: 3.1480 ± 0.7356
	MAE: 2.1289 ± 0.2851
	MAPE:

### ARF (FastODT)

In [9]:
update_freq = "1h"

model = ARFRegressorFODT

param_grid = {
    'n_models': np.arange(10, 40, 10),
    'max_depth': np.arange(50, 300, 50),
    'eagerness': [200],
    'delta': [0.005],
    'aggregation_method' : ["mean"],
    'tau': [0.01],
    'grace_period': [0]
}

best_params, best_scores = grid_search_tuning(param_grid, model, train, valid)

training with: {'n_models': np.int64(10), 'max_depth': np.int64(50), 'eagerness': 200, 'delta': 0.005, 'aggregation_method': 'mean', 'tau': 0.01, 'grace_period': 0}


Processing splits:   0%|          | 0/3 [00:00<?, ?it/s]

Processing devices in split 0:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-09-14 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 1:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-10-12 08:00:00
Train predictions completed.
Validation predictions completed.


... (96 outputs left out)

Processing devices in split 1:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-10-12 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 2:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-11-09 08:00:00
Train predictions completed.
Validation predictions completed.


In [10]:
best_params

{'rmse': {'n_models': np.int64(20),
  'max_depth': np.int64(150),
  'eagerness': 200,
  'delta': 0.005,
  'aggregation_method': 'mean',
  'tau': 0.01,
  'grace_period': 0},
 'mae': {'n_models': np.int64(20),
  'max_depth': np.int64(150),
  'eagerness': 200,
  'delta': 0.005,
  'aggregation_method': 'mean',
  'tau': 0.01,
  'grace_period': 0},
 'mape': {'n_models': np.int64(20),
  'max_depth': np.int64(150),
  'eagerness': 200,
  'delta': 0.005,
  'aggregation_method': 'mean',
  'tau': 0.01,
  'grace_period': 0}}

In [11]:
best_scores

{'rmse': 0.39179826994510486,
 'mae': 0.2360836758658299,
 'mape': 0.5789755835727215}

In [14]:
update_freq = "1h"

best_model_arf_mse = ARFRegressorFODT(**best_params['rmse'])

hoef_all_train_preds, hoef_all_valid_preds, \
hoef_all_train_metrics, hoef_all_valid_metrics = all_valid_splits_seq_fit_predict(
    model=best_model_arf_mse,
    train=train,
    valid=valid,
    incremental=True,
    data_freq=DATA_FREQUENCY,
    update_freq=update_freq,
    start_split=0,
    #end_split=4,  # Use only the third and fourth splits
    verbose=False
)

print("\n\nTrain period metrics:")
hoef_all_time_train_metrics = all_time_aggregation_metrics(hoef_all_train_metrics, data_freq=DATA_FREQUENCY, verbose=False)
print("\n\nValid period metrics:")
hoef_all_time_valid_metrics = all_time_aggregation_metrics(hoef_all_valid_metrics, data_freq=DATA_FREQUENCY, verbose=True)

Processing splits:   0%|          | 0/3 [00:00<?, ?it/s]

Processing devices in split 0:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-09-14 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 1:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-10-12 08:00:00
Train predictions completed.
Validation predictions completed.


Processing devices in split 2:   0%|          | 0/1 [00:00<?, ?it/s]

Initial training on data from 2022-08-18 08:00:00 to 2022-11-09 08:00:00
Train predictions completed.
Validation predictions completed.


Train period metrics:


Valid period metrics:


Aggregated Metrics Across Periods
Device ID: df_TRB145-3
--------------------------------------------------
Hourly Metrics:
	RMSE: 0.3938 ± 0.0428
	MAE: 0.2380 ± 0.0196
	MAPE: 58.5462% ± 10.2207%

	Directional Metrics:
		Directional Accuracy: 60.6653% ± 2.2331%
		First-Diff Correlation: 0.2835 ± 0.0413

	Immediate Detection (delta=0):
		Spike Precision: 0.0000 ± 0.0000
		Spike Recall: 0.0000 ± 0.0000
		Spike F1: 0.0000 ± 0.0000
		Drop Precision: 0.0000 ± 0.0000
		Drop Recall: 0.0000 ± 0.0000
		Drop F1: 0.0000 ± 0.0000

	1-Hour Delayed Detection (delta=1):
		Spike Precision: 0.0000 ± 0.0000
		Spike Recall: 0.0000 ± 0.0000
		Spike F1: 0.0000 ± 0.0000
		Drop Precision: 0.0000 ± 0.0000
		Drop Recall: 0.0000 ± 0.0000
		Drop F1: 0.0000 ± 0.0000
Daily Metrics:
	RMSE: 3.5768 ± 0.7832
	MAE: 2.3748 ± 0.5337
	MAPE